# Validate and publish Gold

Rebuild the complete release, review controls and coverage exceptions, and optionally publish a versioned Gold Delta release to MinIO. This notebook runs independently of notebooks 11–13.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [1]:
from pathlib import Path

from IPython.display import display
import pandas as pd

from gold_pipeline import build, publish

SETTINGS_PATH = Path("gold_settings.json")
gold = build(SETTINGS_PATH, stage="products")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

Silver release 409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5
Cutoff 2026-12-16


,table,rows
0,dim_legal_entity,2
1,dim_customer,5
2,dim_product,5
3,dim_date,14
4,dim_currency,1
5,fact_sales_order_booking,8
6,dim_warehouse,3
7,fact_shipment_line,5
8,dim_document_type,2
9,fact_billing_document,8


In [2]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))

,rule,expected,actual,passed
0,bookings_preserve_eligible_line_count,8,8,True
1,bookings_preserve_eligible_line_amount,220000.000000000000,220000.000000000000,True
2,baseline_booking_rows,2,2,True
3,baseline_distinct_orders,2,2,True
4,baseline_booking_inr,140000,140000.000000000000,True
5,baseline_shared_customer_product,1,1,True
6,baseline_pumps_each,11,11.000000000000,True
7,baseline_net_billing_inr,135000,135000.000000000000,True
8,baseline_gl_net_revenue_inr,135000,135000.000000000000,True
9,baseline_reconciliation_is_matched,True,True,True


,rule,source_id
0,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00002"",""vqk0j2tpqq""]"
1,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00003"",""vqltad6jqq""]"
2,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00004"",""vqlfquq375""]"
3,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00005"",""vqm83vc86c""]"


In [3]:
from gold_delta import publish_to_minio

failed = [row for row in gold.checks if not row["passed"]]
if failed:
    raise ValueError(f"{len(failed)} Gold checks failed. Review the control results above.")

PUBLISH_TARGET = "minio"  # Set to "minio" to commit the validated release.
if PUBLISH_TARGET == "minio":
    gold_manifest_uri = publish_to_minio(gold)
    print("Gold manifest:", gold_manifest_uri)
elif PUBLISH_TARGET == "local":
    output_root = SETTINGS_PATH.resolve().parent / gold.settings["output_root"]
    print((publish(gold, output_root).resolve() / "manifest.json").as_posix())
elif PUBLISH_TARGET == "preview":
    print('Preview complete. Set PUBLISH_TARGET = "minio" to publish Gold Delta tables.')
else:
    raise ValueError('Use "preview", "minio", or "local".')

Gold manifest: s3://otc-gold/releases/f41bae99eb0102db979cbe0dff4c79ed2e2dba4cb13e03800b16091cf3e0d900/manifest.json
